# Orders — Part 3: Building a FHIR Order Message from a CSV

This is the third notebook in the series that started with `01-fhir-search-basics.ipynb`
and `02-work-orders-worked-example.ipynb`. Those two came from the `julius` app and
covered *searching* an existing FHIR server. This notebook switches to this repository's
own pipeline (see `CLAUDE.md`) and covers the other half of the workflow: **building** a
new order from scratch and sending it.

We'll take one row of `Input/NEYctDNA.csv` — an existing test-fixture CSV in this repo —
and turn it into a laboratory order `Bundle` that conforms to the NW-GMSA
[laboratory-order MessageDefinition](https://nw-gmsa.github.io/en/MessageDefinition-laboratory-order.html),
following the shape shown in the IG's own
[Bundle "Message" examples](https://nw-gmsa.github.io/en/StructureDefinition-BundleMessage-examples.html).

The three resources at the heart of any order are:

- [`Patient`](https://nw-gmsa.github.io/en/StructureDefinition-Patient.html) — who the test is for
- [`ServiceRequest`](https://nw-gmsa.github.io/en/StructureDefinition-ServiceRequest.html) — the test being ordered (the message's *focus* resource)
- [`Specimen`](https://nw-gmsa.github.io/en/StructureDefinition-Specimen.html) — the sample the test will be run on

A `MessageHeader` resource wraps these three into a message: it carries the sending and
receiving organisations and points at the `ServiceRequest` as the thing the message is
about. All four resources live in one `Bundle` and reference each other by `fullUrl`
(`urn:uuid:...`) rather than real server URLs, since none of these resources exist on a
server yet.

Once the bundle validates, we'll:

1. Send it to the local interface engine's `$process-message` endpoint, exactly like
   `Testing.ipynb` does for its own order fixtures.
2. Convert it to HL7 v2 `OML^O21` using the `transformToV2` tooling endpoint — the same
   kind of v2 message a real Trust EPR/LIMS would send (see
   [hl7v2.html](https://nw-gmsa.github.io/en/hl7v2.html#oml_o21-laboratory-order)).
3. Compare it with the different shape NHS England's own
   [Genomic Order Management Service FHIR API](https://digital.nhs.uk/developer/api-catalogue/genomic-order-management-service-fhir)
   expects, and submit it there too.

## About the source data

`Input/NEYctDNA.csv` is one of this repo's ctDNA test fixtures (see
`NEYctDNA-HL7v2ORU_R01.ipynb`, which uses the same file to build **report** messages).
Every row already carries a `FillerOrderNumber` and result data — meaning, in real life,
these orders have already been placed and reported on. We're using row 0 (NHS number
`9737873947`, Rheana Newcastle) purely as a source of realistic demographics and test
codes, to *retrospectively* build the order that would have preceded that report — reusing
the same MRN (`RXR3178922` at `RTD`) already recorded for this patient in `MRN-Mapping.md`,
and the same test code already validated for her in `Output/FHIR/R01/ctdna9737873947.txt.json`.
Because this CSV has no `PlacerOrderNumber` column of its own, we synthesise one from
`PatientAccessionIdentifier` (`200041`) where it's used below.

In [1]:
import json
import os
import subprocess
import tempfile
from datetime import datetime, timezone
from uuid import uuid4

import pandas as pd
import requests
from dotenv import load_dotenv
from requests.auth import HTTPBasicAuth

load_dotenv()

toolsServer = os.getenv("V2_TOOLS")    # /transformToFHIR, /transformToV2
fhirServer = os.getenv("FHIR_SERVER")  # the ESB's $process-message endpoint

tokenUrl = os.getenv("OAUTH2_TOKEN")
clientId = os.getenv("CLIENT_ID")
clientSecret = os.getenv("CLIENT_SECRET")

In [2]:
# Same client-credentials exchange Testing.ipynb uses to talk to the same FHIR_SERVER
response = requests.post(
    tokenUrl,
    auth=HTTPBasicAuth(clientId, clientSecret),
    verify=False,
    data={"grant_type": "client_credentials", "scope": "system/*.*"},
    headers={"Content-Type": "application/x-www-form-urlencoded"},
)
token = response.json()["access_token"]
print("Got an access token:", token[:12] + "...")

Got an access token: eyJhbGciOiJS...


/Users/kevinmayfield/github/MFT/Testing/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


## 1. Pick a row from the CSV

`pandas` reads the CSV the same way every other notebook in this repo does
(`NEYctDNA-HL7v2ORU_R01.ipynb`, `CSVTools.ipynb`). We only need the columns relevant to
placing an order — the result/report columns (`ObservationResultStatus`,
`ReportIdentifier`, ...) are left alone.

In [3]:
orders = pd.read_csv("Input/NEYctDNA.csv")
row = orders.iloc[0]

row[[
    "NHSNumber", "HospitalNumber", "PatientFamilyName", "PatientGivenName", "DateOfBirth",
    "AdministrativeSex", "PostCode", "RequestingOrganisationCode", "RequestingOrganisationName",
    "PatientAccessionIdentifier", "FillerOrderNumber", "NGTDTestCode", "NGTDTestName",
    "TestOrderDate", "SpecimenAccessionIdentifier", "SpecimenTypeDescription",
    "SpecimenTakenDateTime", "SpecimenReceivedDateTime",
]]

NHSNumber                                                             9737873947
HospitalNumber                                                        RXR3178922
PatientFamilyName                                                      NEWCASTLE
PatientGivenName                                                          Rheana
DateOfBirth                                                           2009-06-11
AdministrativeSex                                                         Female
PostCode                                                                 NE1 4LP
RequestingOrganisationCode                                                   RTD
RequestingOrganisationName     The Newcastle Upon Tyne Hospitals NHS Foundati...
PatientAccessionIdentifier                                                200041
FillerOrderNumber                                                       R26-15AW
NGTDTestCode                                                               M4.14
NGTDTestName                

## 2. Patient

The [`Patient` profile](https://nw-gmsa.github.io/en/StructureDefinition-Patient.html)
requires at least one identifier, a name, and a `birthDate`; postcode becomes mandatory
once an address is present at all. We give her two identifiers:

- **NHS number** — `system` `https://fhir.nhs.uk/Id/nhs-number`, `type` code `NH`
- **Hospital MRN** — `type` code `MR`, `assigner` the requesting trust (`RTD`)

This is exactly the identifier shape already used for this same patient in
`Output/FHIR/R01/ctdna9737873947.txt.json` — reusing it here keeps her MRN consistent
across fixtures, per this repo's `MRN-Mapping.md` convention.

In [4]:
patient_id = str(uuid4())
patient_fullurl = f"urn:uuid:{patient_id}"

patient = {
    "resourceType": "Patient",
    "identifier": [
        {
            "system": "https://fhir.nhs.uk/Id/nhs-number",
            "type": {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/v2-0203", "code": "NH"}]},
            "value": str(row["NHSNumber"]),
        },
        {
            "assigner": {
                "identifier": {
                    "system": "https://fhir.nhs.uk/Id/ods-organization-code",
                    "value": row["RequestingOrganisationCode"],
                }
            },
            "type": {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/v2-0203", "code": "MR"}]},
            "value": row["HospitalNumber"],
        },
    ],
    "name": [{"family": row["PatientFamilyName"], "given": [row["PatientGivenName"]]}],
    "gender": row["AdministrativeSex"].lower(),
    "birthDate": row["DateOfBirth"],
    "address": [{"postalCode": row["PostCode"]}],
}

print(json.dumps(patient, indent=2))

{
  "resourceType": "Patient",
  "identifier": [
    {
      "system": "https://fhir.nhs.uk/Id/nhs-number",
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "NH"
          }
        ]
      },
      "value": "9737873947"
    },
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "RTD"
        }
      },
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "MR"
          }
        ]
      },
      "value": "RXR3178922"
    }
  ],
  "name": [
    {
      "family": "NEWCASTLE",
      "given": [
        "Rheana"
      ]
    }
  ],
  "gender": "female",
  "birthDate": "2009-06-11",
  "address": [
    {
      "postalCode": "NE1 4LP"
    }
  ]
}


## 3. Validate the Patient resource on its own

[testing.html](https://nw-gmsa.github.io/en/testing.html) recommends validating
individual resources before validating a whole bundle — "the FHIR Validator defaults to
validating individual FHIR resources (not FHIR Bundles)". For a single resource that
means passing `-profile <canonical-url>` instead of the `-bundle` flag `FHIR
Validation.ipynb` uses.

We write each candidate resource to a throwaway temp file for this — these are
dev-loop checks, not fixtures we're keeping, unlike the final bundle further down which
does get saved permanently.

In [5]:
PATIENT_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/Patient"
SERVICE_REQUEST_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/ServiceRequest"
SPECIMEN_PROFILE = "https://fhir.nwgenomics.nhs.uk/StructureDefinition/Specimen"


def details(item):
    return item["text"]


def issues_df(outcome):
    df = pd.DataFrame(outcome["issue"])
    df["details"] = df["details"].apply(details)
    df.drop(columns=["extension"], inplace=True, errors="ignore")
    df.sort_values(by=["severity"], inplace=True)
    # same noisy, expected-in-this-IG warnings FHIR Validation.ipynb filters out
    df = df[~df["details"].str.contains("ValueSet/mimetypes")]
    df = df[~df["details"].str.contains("failed: dom-6")]
    df = df[~df["details"].str.contains("bcp:13")]
    return df


def validate_resource(resource, profile):
    # Validate a single (non-bundled) resource against one NW-GMSA profile.
    with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as tmp:
        json.dump(resource, tmp)
        tmp_path = tmp.name
    outcome_path = tmp_path + "-OperationOutcome.json"

    # capture_output swallows the validator's verbose IG/package loading log -
    # only the resulting OperationOutcome (read back below) matters here
    subprocess.run(
        [
            "java", "-jar", "validator_cli.jar", tmp_path,
            "-version", "4.0.1", "-ig", "package.tgz",
            "-profile", profile, "-tx", "n/a",
            "-output", outcome_path, "-output-style", "json",
        ],
        capture_output=True,
    )

    with open(outcome_path) as f:
        outcome = json.load(f)
    return issues_df(outcome)


validate_resource(patient, PATIENT_PROFILE)

,severity,code,details,expression


## 4. Specimen

The [`Specimen` profile](https://nw-gmsa.github.io/en/StructureDefinition-Specimen.html)
requires a `type` and a `subject`. The CSV only gives us a free-text description
(`SpecimenTypeDescription` = "Blood"), so we map it to the same SNOMED code the real
transformation engine already produced for this test type elsewhere in this repo —
`258580003` "Whole blood specimen" (see `Output/FHIR/O21/OML_O21_R0A_R125.txt.json`).

`subject` carries both a `reference` to the `Patient` entry above *and* a logical
`identifier` — the same belt-and-braces pattern used throughout this IG's own generated
examples, so the reference still resolves even for a consumer that only looks up
patients by NHS number.

In [6]:
def to_fhir_datetime(value):
    # CSV stores e.g. "29 Jun 2026 12:00"; NW-GMSA examples use ISO 8601 with an offset
    return datetime.strptime(value, "%d %b %Y %H:%M").strftime("%Y-%m-%dT%H:%M:%S+00:00")


specimen_id = str(uuid4())
specimen_fullurl = f"urn:uuid:{specimen_id}"

specimen = {
    "resourceType": "Specimen",
    "identifier": [
        {
            "assigner": {
                "identifier": {
                    "system": "https://fhir.nhs.uk/Id/ods-organization-code",
                    "value": row["RequestingOrganisationCode"],
                }
            },
            "type": {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/v2-0203", "code": "PLAC"}]},
            "value": row["SpecimenAccessionIdentifier"],
        }
    ],
    "status": "available",
    "type": {
        "coding": [{"system": "http://snomed.info/sct", "code": "258580003", "display": "Whole blood specimen"}]
    },
    "subject": {
        "reference": patient_fullurl,
        "identifier": {
            "system": "https://fhir.nhs.uk/Id/nhs-number",
            "type": {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/v2-0203", "code": "NH"}]},
            "value": str(row["NHSNumber"]),
        },
    },
    "collection": {"collectedDateTime": to_fhir_datetime(row["SpecimenTakenDateTime"])},
    "receivedTime": to_fhir_datetime(row["SpecimenReceivedDateTime"]),
}

print(json.dumps(specimen, indent=2))

{
  "resourceType": "Specimen",
  "identifier": [
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "RTD"
        }
      },
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "PLAC"
          }
        ]
      },
      "value": "S26-1K1L"
    }
  ],
  "status": "available",
  "type": {
    "coding": [
      {
        "system": "http://snomed.info/sct",
        "code": "258580003",
        "display": "Whole blood specimen"
      }
    ]
  },
  "subject": {
    "reference": "urn:uuid:3948ed46-472e-42c8-9437-d4b2554044b8",
    "identifier": {
      "system": "https://fhir.nhs.uk/Id/nhs-number",
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "NH"
          }
        ]
      },
      "value": "9737873947"
    }
  },
  "collecti

In [7]:
validate_resource(specimen, SPECIMEN_PROFILE)

,severity,code,details,expression
0,information,code-invalid,Could not confirm that the codes provided are ...,[Specimen.type]


The one `information`-level `code-invalid` result above ("Could not confirm that
the codes provided are correct") is the validator telling us it can't check the SNOMED
code against a real terminology server — expected, since we passed `-tx n/a`
(`testing.html` notes the validator can't act as a terminology server against NHS
England's Ontology Service). Not a defect in the resource.

## 5. ServiceRequest — the message's focus resource

The [`ServiceRequest` profile](https://nw-gmsa.github.io/en/StructureDefinition-ServiceRequest.html)
is the resource `MessageHeader.focus` will point at. Key fields:

- `status`/`intent` — `active`/`order`: this order is in flight (contrast with the
  `completed` `ServiceRequest` already sitting in the R01 report fixture for this patient)
- `code` — the test from the England Genomic Test Directory (`M4.14`, `NGTDTestCode`)
- `category` — SNOMED `116148004`, the same category coding used throughout this repo's
  other O21/R01 examples
- `identifier` — **two** order numbers, matching ORC-2/ORC-3 in the v2 mapping below:
  - `PLAC` (placer order number), assigned by the ordering trust. This CSV has no
    `PlacerOrderNumber` of its own (see the intro), so we stand in with
    `PatientAccessionIdentifier`.
  - `FILL` (filler order number) — `FillerOrderNumber` from the CSV, assigned by the GLH
    (`699X0`, the same ODS code used as the GLH assigner throughout this repo's other
    O21/R01 fixtures)
- `requester` — the ordering trust, as an `Organization`
- `specimen` — a reference to the `Specimen` entry above

In [8]:
service_request_id = str(uuid4())
service_request_fullurl = f"urn:uuid:{service_request_id}"

service_request = {
    "resourceType": "ServiceRequest",
    "status": "active",
    "intent": "order",
    "category": [{"coding": [{"system": "http://snomed.info/sct", "code": "116148004"}]}],
    "code": {
        "coding": [{"system": "https://fhir.nhs.uk/CodeSystem/England-GenomicTestDirectory", "code": row["NGTDTestCode"]}]
    },
    "identifier": [
        {
            "assigner": {
                "identifier": {
                    "system": "https://fhir.nhs.uk/Id/ods-organization-code",
                    "value": row["RequestingOrganisationCode"],
                }
            },
            "type": {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/v2-0203", "code": "PLAC"}]},
            "value": str(row["PatientAccessionIdentifier"]),  # stand-in - see intro
        },
        {
            "assigner": {
                "identifier": {"system": "https://fhir.nhs.uk/Id/ods-organization-code", "value": "699X0"}
            },
            "type": {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/v2-0203", "code": "FILL"}]},
            "value": row["FillerOrderNumber"],
        },
    ],
    "subject": {
        "reference": patient_fullurl,
        "identifier": {
            "system": "https://fhir.nhs.uk/Id/nhs-number",
            "type": {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/v2-0203", "code": "NH"}]},
            "value": str(row["NHSNumber"]),
        },
    },
    "requester": {
        "display": row["RequestingOrganisationName"],
        "identifier": {"system": "https://fhir.nhs.uk/Id/ods-organization-code", "value": row["RequestingOrganisationCode"]},
        "type": "Organization",
    },
    "specimen": [{"reference": specimen_fullurl, "type": "Specimen"}],
    "authoredOn": to_fhir_datetime(row["TestOrderDate"]),
}

print(json.dumps(service_request, indent=2))

{
  "resourceType": "ServiceRequest",
  "status": "active",
  "intent": "order",
  "category": [
    {
      "coding": [
        {
          "system": "http://snomed.info/sct",
          "code": "116148004"
        }
      ]
    }
  ],
  "code": {
    "coding": [
      {
        "system": "https://fhir.nhs.uk/CodeSystem/England-GenomicTestDirectory",
        "code": "M4.14"
      }
    ]
  },
  "identifier": [
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "RTD"
        }
      },
      "type": {
        "coding": [
          {
            "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
            "code": "PLAC"
          }
        ]
      },
      "value": "200041"
    },
    {
      "assigner": {
        "identifier": {
          "system": "https://fhir.nhs.uk/Id/ods-organization-code",
          "value": "699X0"
        }
      },
      "type": {
        "coding": [
          {
    

In [9]:
validate_resource(service_request, SERVICE_REQUEST_PROFILE)

,severity,code,details,expression
0,information,informational,This element does not match any known slice de...,[ServiceRequest.identifier[1]]
3,information,code-invalid,None of the codings provided are in the value ...,[ServiceRequest.code]
1,warning,code-invalid,Unknown Code 'M4.14' in the CodeSystem 'https:...,[ServiceRequest.code.coding[0].code]
2,warning,not-found,A definition for the value Set 'http://hl7.org...,[ServiceRequest.code]


Same story here — the `code-invalid`/`not-found` warnings are the validator unable
to check `M4.14` against the Genomic Test Directory value set without a terminology
server, and the `identifier[1]` "doesn't match any known slice" hint is the validator
flagging that it can't tell our `FILL` identifier apart from a differently-sliced one
without more context. These exact same warnings already show up against the
`ServiceRequest` in this repo's own previously-validated
`Output/FHIR/O21/OML_O21_R0A_R125.txt.json` fixture (see
`Results/FHIR/O21/OML_O21_R0A_R125.txt.json-OperationOutcome.json`) — expected noise for
this IG under `-tx n/a`, not something specific to this order.

## 6. MessageHeader — wrapping it all into a message

`MessageHeader` doesn't have its own NW-GMSA profile page to validate against
individually — it's checked implicitly when we validate the whole bundle next. It
carries:

- `eventCoding` — `O21`, `http://terminology.hl7.org/CodeSystem/v2-0003` (the FHIR
  equivalent of MSH-9 in the v2 message)
- `sender` — the ordering trust
- `destination` — the receiving GLH (`699X0`)
- `focus` — a reference to the `ServiceRequest`, per the
  [MessageDefinition](https://nw-gmsa.github.io/en/MessageDefinition-laboratory-order.html)

In [10]:
message_header = {
    "resourceType": "MessageHeader",
    "eventCoding": {"system": "http://terminology.hl7.org/CodeSystem/v2-0003", "code": "O21"},
    "sender": {
        "identifier": {"system": "https://fhir.nhs.uk/Id/ods-organization-code", "value": row["RequestingOrganisationCode"]}
    },
    "destination": [
        {
            "endpoint": "https://fhir.nwgenomics.nhs.uk/Endpoint/GLH",
            "receiver": {"identifier": {"system": "https://fhir.nhs.uk/Id/ods-organization-code", "value": "699X0"}},
        }
    ],
    "source": {"endpoint": fhirServer},
    "focus": [{"reference": service_request_fullurl}],
}

## 7. Assemble and validate the whole Bundle

`Bundle.type` is `message` (not `collection` or `transaction` — more on that
distinction in section 10). `MessageHeader` comes first, matching every NW-GMSA order
example already in this repo (`Output/FHIR/O21/OML_O21_R0A_R125.txt.json`).

The [`Bundle` (message) profile](https://nw-gmsa.github.io/en/StructureDefinition-BundleMessage.html)
this Bundle conforms to also makes two top-level elements mandatory, easy to miss since
they sit alongside `entry` rather than inside any one resource:

- `Bundle.identifier` — a business identifier for the bundle as a whole (distinct from
  any resource's own identifiers). The existing fixtures in this repo shape it as a bare
  `urn:uuid:...` value with no separate `system` (see
  `Output/FHIR/O21/OML_O21_R0A_R125.txt.json`), so we follow that.
- `Bundle.timestamp` — when the bundle was assembled.

We'll reuse `Bundle.identifier.value` again in section 11, as NHS England's
`X-Correlation-ID` header.

Unlike the throwaway per-resource files above, this bundle is worth keeping: per this
repo's own directory layout (`CLAUDE.md`), hand-built FHIR examples destined for
`transformToV2` live in `Input/FHIR/<type>/`.

In [11]:
bundle_identifier = f"urn:uuid:{uuid4()}"

order_bundle = {
    "resourceType": "Bundle",
    "identifier": {"value": bundle_identifier},
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S+00:00"),
    "type": "message",
    "entry": [
        {"fullUrl": f"urn:uuid:{uuid4()}", "resource": message_header},
        {"fullUrl": patient_fullurl, "resource": patient},
        {"fullUrl": specimen_fullurl, "resource": specimen},
        {"fullUrl": service_request_fullurl, "resource": service_request},
    ],
}

order_filename = f"NEYctDNA_Order_{row['NHSNumber']}.json"
with open("Input/FHIR/O21/" + order_filename, "w") as f:
    json.dump(order_bundle, f, indent=2)

print("Saved Input/FHIR/O21/" + order_filename)

Saved Input/FHIR/O21/NEYctDNA_Order_9737873947.json


In [12]:
folder = "FHIR/O21/"

subprocess.run(
    [
        "java", "-jar", "validator_cli.jar", "Input/" + folder + order_filename,
        "-version", "4.0.1", "-ig", "package.tgz",
        "-bundle", "ServiceRequest:0", SERVICE_REQUEST_PROFILE, "-tx", "n/a",
        "-output", "Results/" + folder + order_filename + "-OperationOutcome.json",
        "-output-style", "json",
    ],
    capture_output=True,
)

with open("Results/" + folder + order_filename + "-OperationOutcome.json") as f:
    outcome = json.load(f)

issues_df(outcome)

,severity,code,details,expression
5,information,informational,This element does not match any known slice de...,[Bundle.entry[3].resource.identifier[1]]
7,information,code-invalid,None of the codings provided are in the value ...,[Bundle.entry[3].resource.code]
3,warning,code-invalid,Unknown Code 'M4.14' in the CodeSystem 'https:...,[Bundle.entry[3].resource/*ServiceRequest/null...
6,warning,not-found,A definition for the value Set 'http://hl7.org...,[Bundle.entry[3].resource.code]


## 8. Send the order — `$process-message`

Exactly the same call `Testing.ipynb` makes for its own O21 fixtures: `POST` the bundle
to the ESB's `$process-message` endpoint (`FHIR_SERVER` in `.env`), bearer-authenticated
with the OAuth2 token from step 0. This is what routes the order onward and updates the
FHIR repository.

In [13]:
headersFHIR = {"Content-Type": "application/fhir+json", "Authorization": "Bearer " + token}

with open("Input/FHIR/O21/" + order_filename, "rb") as f:
    order_json = f.read()

response = requests.post(fhirServer + "$process-message", verify=False, data=order_json, headers=headersFHIR)
print(response.status_code)
print(response.text[:2000])

/Users/kevinmayfield/github/MFT/Testing/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


200
{
  "resourceType": "Bundle",
  "entry": [
    {
      "fullUrl": "urn:uuid:796a03f7-a59b-4354-b375-7ef30a63135c",
      "resource": {
        "resourceType": "MessageHeader",
        "destination": [
          {
            "receiver": {
              "identifier": {
                "system": "https://fhir.nhs.uk/Id/ods-organization-code",
                "value": "RTD"
              }
            }
          }
        ],
        "eventCoding": {
          "code": "O21",
          "system": "http://terminology.hl7.org/CodeSystem/v2-0003"
        },
        "response": {
          "code": "ok",
          "details": {
            "reference": "urn:uuid:2ea9071f-41bf-4bbc-9c30-f1b6c9acaa66"
          },
          "identifier": "urn:uuid:089b27eb-1b44-4f1b-8de9-1e8ea6a0f7d3"
        },
        "sender": {
          "identifier": {
            "system": "https://fhir.nhs.uk/Id/ods-organization-code",
            "value": "X24"
          }
        },
        "source": {
          "endpo

## 9. Convert it to HL7 v2 `OML^O21`

This message is deliberately backwards-compatible with HL7 v2, the format most NHS
Trust EPRs and LIMS still speak for orders (see
[hl7v2.html](https://nw-gmsa.github.io/en/hl7v2.html#oml_o21-laboratory-order)). North
West Genomics uses v2.5.1 `OML^O21` rather than the older `ORM^O01` (`Testing.ipynb`
converts any `O01` fixtures it receives into `O21` for exactly this reason).

Rather than write the v2 by hand, we ask the same `transformToV2` tooling endpoint
`Testing.ipynb` already uses for its own O21 fixtures to do the conversion:

| HL7 v2 segment | FHIR resource |
|---|---|
| MSH | MessageHeader |
| PID | Patient |
| ORC / OBR | ServiceRequest |
| SPM | Specimen |

In [14]:
rV2 = requests.post(toolsServer + "/transformToV2", data=order_json, verify=False, headers=headersFHIR)

v2_filename = order_filename.replace(".json", ".txt")
with open("Output/V2/O21/" + v2_filename, "w") as f:
    f.write(rV2.text)

print(rV2.text)

## 10. NW-GMSA's `message` Bundle vs NHS England's `transaction` Bundle

NHS England's own national
[Genomic Order Management Service FHIR API](https://digital.nhs.uk/developer/api-catalogue/genomic-order-management-service-fhir)
models an order submission differently: `POST /FHIR/R4` expects a **transaction**
`Bundle` (`Bundle.type = "transaction"`) — a flat batch of independent REST operations,
each entry carrying its own `request.method`/`request.url` — rather than a `message`
Bundle with a `MessageHeader` envelope.

NW-GMSA chose the `message` shape instead because it:

- has a natural "event" (`eventCoding`) to carry the v2 `OML^O21`/`ORM^O01` equivalent of
  MSH-9, which is what makes the `transformToV2` conversion above possible
- carries `sender`/`destination`, which the interface engine uses for routing
- is a superset of the data a `transaction` Bundle needs — it can still be converted to
  one for systems, like NHS England's, that expect it

The conversion just drops the `MessageHeader` entry and turns each remaining entry into
a `POST` of its own resource type:

In [15]:
def to_transaction_bundle(message_bundle):
    entries = []
    for entry in message_bundle["entry"]:
        resource = entry["resource"]
        if resource["resourceType"] == "MessageHeader":
            continue  # a transaction Bundle has no envelope resource
        entries.append({
            "fullUrl": entry["fullUrl"],
            "resource": resource,
            "request": {"method": "POST", "url": resource["resourceType"]},
        })
    return {"resourceType": "Bundle", "type": "transaction", "entry": entries}


transaction_bundle = to_transaction_bundle(order_bundle)
print(json.dumps(transaction_bundle, indent=2))

{
  "resourceType": "Bundle",
  "type": "transaction",
  "entry": [
    {
      "fullUrl": "urn:uuid:3948ed46-472e-42c8-9437-d4b2554044b8",
      "resource": {
        "resourceType": "Patient",
        "identifier": [
          {
            "system": "https://fhir.nhs.uk/Id/nhs-number",
            "type": {
              "coding": [
                {
                  "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
                  "code": "NH"
                }
              ]
            },
            "value": "9737873947"
          },
          {
            "assigner": {
              "identifier": {
                "system": "https://fhir.nhs.uk/Id/ods-organization-code",
                "value": "RTD"
              }
            },
            "type": {
              "coding": [
                {
                  "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
                  "code": "MR"
                }
              ]
            },
            "

## 11. Submit the transaction to NHS England (integration environment)

`int.api.service.nhs.uk` is NHS England's **integration** (sandbox) tier for developer
testing — not the live service — so it's safe to post this synthetic order to it. The
endpoint and `apikey` live in `.env` (`NHSE_GENOMIC_ORDER_API` /
`NHSE_GENOMIC_ORDER_APIKEY`, gitignored) rather than in this notebook, so the key is
never committed. Alongside the `apikey`, we send two more headers NHS England APIs
conventionally expect:

- `X-Request-ID` — a fresh UUID per request, unrelated to the bundle itself.
- `X-Correlation-ID` — ties this specific submission back to the order it's carrying.
  We reuse `Bundle.identifier.value` from the original `message` Bundle (section 7) for
  this, rather than generating another new id.

The OAS linked from the API catalogue page above is the authoritative contract for the
exact headers and any conditional-create semantics this specific API expects; this is
illustrative.

In [16]:
nhseUrl = os.getenv("NHSE_GENOMIC_ORDER_API")
nhseApiKey = os.getenv("NHSE_GENOMIC_ORDER_APIKEY")

headersNHSE = {
    "Content-Type": "application/fhir+json",
    "Accept": "application/fhir+json",
    "apikey": nhseApiKey,
    "X-Request-ID": str(uuid4()),
    "X-Correlation-ID": bundle_identifier,
}

responseNHSE = requests.post(nhseUrl, data=json.dumps(transaction_bundle), headers=headersNHSE)
print(responseNHSE.status_code)
print(responseNHSE.text[:2000])

401



A `401` with an empty body (as above) is a realistic outcome to get back the first
time against a real NHS England API: it means the gateway rejected the request before it
ever reached the FHIR facade, almost always an auth problem rather than anything wrong
with the bundle itself. Most NHS England Apigee-fronted APIs beyond simple read-only
endpoints expect a full OAuth2 flow (a signed JWT client assertion exchanged for a
bearer token) layered on top of the `apikey`, not just the `apikey` header on its own —
exactly the kind of detail the OAS linked above is the authority on, and worth checking
before assuming the request itself is wrong.

## Summary

Starting from one row of an existing test-fixture CSV, we built and validated a
`Patient`/`Specimen`/`ServiceRequest` order `Bundle` conforming to the NW-GMSA
laboratory-order `MessageDefinition`, sent it through the local interface engine,
round-tripped it to HL7 v2 `OML^O21`, and compared/submitted it as an NHS England-style
`transaction` Bundle. The same `Input/FHIR/O21/` bundle this notebook saved can now be
picked up by `Testing.ipynb` and `FHIR Validation.ipynb` like any other fixture in this
repo.